In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import time

In [68]:
load_dotenv()
API_KEY = os.getenv('API_KEY')
print(API_KEY)

bbc56ce0260a4036ba8750315199a7d1


In [59]:

API_KEY = os.getenv('API_KEY')
BLS_V2_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"  


state_fips = [
    '01','02','04','05','06','08','09','10','12','13',
    '15','16','17','18','19','20','21','22','23','24',
    '25','26','27','28','29','30','31','32','33','34',
    '35','36','37','38','39','40','41','42','44','45',
    '46','47','48','49','50','51','53','54','55','56'
]
state_names = [
    'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
    'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
    'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
    'Maine','Maryland','Massachusetts','Michigan','Minnesota',
    'Mississippi','Missouri','Montana','Nebraska','Nevada',
    'New Hampshire','New Jersey','New Mexico','New York',
    'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
    'Pennsylvania','Rhode Island','South Carolina','South Dakota',
    'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
    'West Virginia','Wisconsin','Wyoming'
]

# saved_state = [    
#     'Alabama',  'California',    'Delaware',     'Florida',       'Idaho',
#     'Iowa',      'Kansas',       'Maine',    'Michigan',   'Minnesota',
#     'Puerto Rico'
# ]

# for name in state_names:
#     for name2 in saved_state:
#         if name == name2:
#             new_list= 



sectors = {
    '06': 'Goods_Producing',
    '07': 'Service_Providing',
    '10': 'Mining_and_Logging',
    '20': 'Construction',
    '30': 'Manufacturing',
    '40': 'Trade_Transport_and_Utilities',
    '50': 'Information',
    '55': 'Financial_Activities',
    '60': 'Professional_and_Tech_Services',
    '65': 'Education_and_Health_Services',
    '70': 'Leisure_and_Hospitality',
    '90': 'Government',
}

# v2 limits: 50 series/request, 20 years/request
# 2016-2026 fits in one window; split if you ever go beyond 20 years
TIME_PERIODS  = [("2016", "2026")]
BATCH_SIZE    = 50




def chunked(lst: list, size: int):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def safe_float(value: str):
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def fetch_bls_v2(series_batch: list[str], start_year: str, end_year: str) -> dict:
    """POST a batch of series to the BLS v2 API and return parsed JSON."""
    payload = {
        "seriesid":        series_batch,
        "startyear":       start_year,       
        "endyear":         end_year,          
        "registrationkey": API_KEY,           
        "catalog":         False,
        "calculations":    False,
        "annualaverage":   False,             
    }
    headers  = {"Content-type": "application/json"}
    response = requests.post(BLS_V2_URL, data=json.dumps(payload), headers=headers)  
    response.raise_for_status()
    return response.json()


def parse_bls_response(json_data: dict, id_to_meta: dict) -> list[dict]:
    """Extract monthly records from a BLS API response."""
    records = []
    status  = json_data.get("status")
    if status != "REQUEST_SUCCEEDED":
        print(f"  API warning ({status}): {json_data.get('message', 'Unknown error')}")
        # if "Results" not in json_data:
        #     return records  

    for series in json_data["Results"]["series"]:
        sid  = series["seriesID"]
        meta = id_to_meta.get(sid, {})
        for item in series["data"]:
            period = item["period"]          
            if "M01" <= period <= "M12":
                records.append({
                    **meta,
                    "Year":  int(item["year"]),
                    "Month": int(period.replace("M", "")),
                    "Value": safe_float(item["value"]),
                })
    return records


# STEP 1: STATE UNEMPLOYMENT (LAUS) 
print("=" * 60)
print("STEP 1: Fetching state unemployment rates (LAUS)...")
print("=" * 60)

laus_series = [f"LASST{fips}0000000000003" for fips in state_fips]
laus_meta   = {
    f"LASST{fips}0000000000003": {"State": name, "Metric": "Unemployment_Rate"}
    for fips, name in zip(state_fips, state_names)   
}

laus_records = []
for start, end in TIME_PERIODS:
    for batch in chunked(laus_series, BATCH_SIZE):
        print(f"  LAUS: {len(batch)} states | {start}–{end}")
        result = fetch_bls_v2(batch, start, end)
        laus_records.extend(parse_bls_response(result, laus_meta))
        time.sleep(0.5)

df_laus = pd.DataFrame(laus_records)[["State", "Year", "Month", "Value"]]
df_laus = df_laus.rename(columns={"Value": "Unemployment_Rate"})
print(f"  ✓ {len(df_laus):,} LAUS records fetched")


# STEP 2: SECTOR EMPLOYMENT (SMU) 
print("\n" + "=" * 60)
print("STEP 2: Fetching sector employment (SMU)...")
print(f"  {len(state_fips)} states × {len(sectors)} sectors = {len(state_fips)*len(sectors)} series")
print("=" * 60)

smu_series = []
smu_meta   = {}
for fips, state_name in zip(state_fips, state_names):
    for sector_code, sector_name in sectors.items():
        sid = f"SMU{fips}000000{sector_code}0000001"
        smu_series.append(sid)
        smu_meta[sid] = {"State": state_name, "Sector": sector_name}

smu_records  = []
total_batches = len(list(chunked(smu_series, BATCH_SIZE))) * len(TIME_PERIODS)
batch_count   = 0

for start, end in TIME_PERIODS:
    for batch in chunked(smu_series, BATCH_SIZE):
        batch_count += 1
        print(f"  Batch {batch_count}/{total_batches} | {len(batch)} series | {start}–{end}")
        result = fetch_bls_v2(batch, start, end)
        smu_records.extend(parse_bls_response(result, smu_meta))
        time.sleep(0.5)

df_smu = pd.DataFrame(smu_records)
print(f"  ✓ {len(df_smu):,} SMU records fetched")


#  STEP 3: PIVOT SECTORS INTO COLUMNS 
print("\n" + "=" * 60)
print("STEP 3: Pivoting sectors into columns...")
print("=" * 60)

df_sectors = df_smu.pivot_table(
    index   = ["State", "Year", "Month"],
    columns = "Sector",
    values  = "Value",
    aggfunc = "first",
).reset_index()
df_sectors.columns.name = None


#  STEP 4: MERGE 
print("\n" + "=" * 60)
print("STEP 4: Merging unemployment + sectors...")
print("=" * 60)

df_merged = df_laus.merge(df_sectors, on=["State", "Year", "Month"], how="left")

df_merged["Date"] = pd.to_datetime(
    df_merged[["Year", "Month"]].rename(
        columns={"Year": "year", "Month": "month"}
    ).assign(day=1)
)

df_merged["Total_Tech_Employment_Thousands"] = (
    df_merged.get("Information",                    pd.Series(0, index=df_merged.index)).fillna(0) +
    df_merged.get("Professional_and_Tech_Services", pd.Series(0, index=df_merged.index)).fillna(0)
)

sector_cols = list(sectors.values())
col_order   = (
    ["State", "Date", "Year", "Month", "Unemployment_Rate"]
    + [c for c in sector_cols if c in df_merged.columns]
    + ["Total_Tech_Employment_Thousands"]
)
df_merged = df_merged[col_order].sort_values(["State", "Year", "Month"]).reset_index(drop=True)


#  STEP 5: SUMMARY & EXPORT 
print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Total records  : {len(df_merged):,}")
print(f"  States         : {df_merged['State'].nunique()}")
print(f"  Sectors        : {len(sectors)}")
print(f"  Date range     : {df_merged['Date'].min().strftime('%b %Y')} → {df_merged['Date'].max().strftime('%b %Y')}")
print(f"\nMissing values per column:")
print(df_merged.isna().sum().to_string())
print(f"\nSample (first 5 rows):")
print(df_merged.head())

out_file = "state_unemployment_all_sectors_2016_2026_v2.csv"
df_merged.to_csv(out_file, index=False)
print(f"\n✓ Saved: {out_file}")

STEP 1: Fetching state unemployment rates (LAUS)...
  LAUS: 50 states | 2016–2026
  API warning (REQUEST_NOT_PROCESSED): ['Request could not be serviced, as the daily threshold for total number of requests allocated to the user with registration key null has been reached.']


KeyError: 'series'